# Kalshi Market Time Series Construction
Process all markets & trades parquet files to build bid-ask spread and last price time series.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

PATH = "../data/kalshi"
MARKETS_DIR = os.path.join(PATH, "markets")
TRADES_DIR = os.path.join(PATH, "trades")
OUTPUT_DIR = os.path.join(PATH, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

market_files = sorted(glob.glob(os.path.join(MARKETS_DIR, "*.parquet")))
trade_files = sorted(glob.glob(os.path.join(TRADES_DIR, "*.parquet")))
print(f"Market files: {len(market_files)}, Trade files: {len(trade_files)}")

## 1. Process Markets — extract bid/ask/spread/last_price per ticker over time
We process in chunks to avoid OOM, keeping only the columns we need.

In [ ]:
MARKET_COLS = ["ticker", "yes_bid", "yes_ask", "no_bid", "no_ask", "last_price", "open_time", "close_time", "status", "volume"]

chunks = []
for i, f in enumerate(market_files):
    df = pd.read_parquet(f, columns=MARKET_COLS)
    chunks.append(df)
    if (i + 1) % 100 == 0:
        print(f"  loaded {i+1}/{len(market_files)} market files")

markets = pd.concat(chunks, ignore_index=True)
del chunks
print(f"Total market rows: {len(markets):,}")
markets.head()

In [ ]:
# Compute spreads
markets["yes_spread"] = markets["yes_ask"] - markets["yes_bid"]
markets["no_spread"] = markets["no_ask"] - markets["no_bid"]

# Parse times
markets["open_time"] = pd.to_datetime(markets["open_time"], utc=True)
markets["close_time"] = pd.to_datetime(markets["close_time"], utc=True)

markets.head()

In [ ]:
# Save full processed markets table
markets.to_parquet(os.path.join(OUTPUT_DIR, "all_markets.parquet"), index=False)
print("Saved all_markets.parquet")

## 2. Process Trades — last price time series from actual trade executions

In [ ]:
TRADE_COLS = ["ticker", "yes_price", "no_price", "count", "taker_side", "created_time"]

# Trades are ~72M rows — process in batches and write intermediate parquets
BATCH = 500
batch_paths = []

for batch_start in range(0, len(trade_files), BATCH):
    batch_end = min(batch_start + BATCH, len(trade_files))
    chunks = []
    for f in trade_files[batch_start:batch_end]:
        chunks.append(pd.read_parquet(f, columns=TRADE_COLS))
    batch_df = pd.concat(chunks, ignore_index=True)
    batch_df["created_time"] = pd.to_datetime(batch_df["created_time"], utc=True)
    out = os.path.join(OUTPUT_DIR, f"trades_batch_{batch_start}.parquet")
    batch_df.to_parquet(out, index=False)
    batch_paths.append(out)
    print(f"  batch {batch_start}-{batch_end}: {len(batch_df):,} trades")
    del chunks, batch_df

print(f"Wrote {len(batch_paths)} trade batches")

In [ ]:
# Combine all trade batches
trades = pd.concat([pd.read_parquet(p) for p in batch_paths], ignore_index=True)
trades.sort_values("created_time", inplace=True)
trades.reset_index(drop=True, inplace=True)
print(f"Total trade rows: {len(trades):,}")
trades.head()

In [ ]:
trades.to_parquet(os.path.join(OUTPUT_DIR, "all_trades.parquet"), index=False)
print("Saved all_trades.parquet")

# Clean up batch files
for p in batch_paths:
    os.remove(p)

## 3. Build per-ticker time series
For each ticker, merge the snapshot-level bid/ask from markets with the trade-level last prices.

In [ ]:
# Aggregate trades to get last price time series per ticker
# Resample to 1-minute OHLC of yes_price per ticker
trades.set_index("created_time", inplace=True)

trade_ts = (
    trades.groupby("ticker")["yes_price"]
    .resample("1min")
    .agg(["last", "mean", "count"])
    .dropna(subset=["last"])
    .rename(columns={"last": "last_yes_price", "mean": "vwap_yes", "count": "n_trades"})
)

trade_ts.reset_index(inplace=True)
print(f"Trade time series rows: {len(trade_ts):,}")
trade_ts.head(10)

In [ ]:
trade_ts.to_parquet(os.path.join(OUTPUT_DIR, "trade_timeseries_1min.parquet"), index=False)
print("Saved trade_timeseries_1min.parquet")

## 4. Quick summary stats & sanity checks

In [ ]:
print("=== Markets ===")
print(f"Unique tickers: {markets['ticker'].nunique():,}")
print(f"Date range: {markets['open_time'].min()} -> {markets['close_time'].max()}")
print(f"\nYes spread stats:")
print(markets["yes_spread"].describe())
print(f"\nNo spread stats:")
print(markets["no_spread"].describe())

print("\n=== Trades ===")
print(f"Unique tickers traded: {trades['ticker'].nunique():,}")
print(f"Date range: {trades.index.min()} -> {trades.index.max()}")

In [ ]:
# Example: plot spread & last price for a single active ticker
import matplotlib.pyplot as plt

# Pick ticker with most trades
top_ticker = trades.groupby("ticker").size().idxmax()
print(f"Most traded ticker: {top_ticker}")

ts = trade_ts[trade_ts["ticker"] == top_ticker].set_index("created_time")
mkt = markets[markets["ticker"] == top_ticker].iloc[0]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(ts.index, ts["last_yes_price"], lw=0.8)
axes[0].set_ylabel("Yes Price (cents)")
axes[0].set_title(f"{top_ticker} — Last Trade Price")

axes[1].bar(ts.index, ts["n_trades"], width=0.0007, alpha=0.7)
axes[1].set_ylabel("# Trades (1min)")
axes[1].set_title("Trade Volume")

plt.tight_layout()
plt.show()

print(f"\nMarket snapshot — yes_bid: {mkt['yes_bid']}, yes_ask: {mkt['yes_ask']}, yes_spread: {mkt['yes_spread']}")
print(f"                  no_bid: {mkt['no_bid']}, no_ask: {mkt['no_ask']}, no_spread: {mkt['no_spread']}")